In [153]:
import numpy as np
import pickle
import matplotlib.pyplot as plt

In [154]:
class NumpyCompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.startswith("numpy._core"):
            module = module.replace("numpy._core", "numpy.core")
        return super().find_class(module, name)
if np.__version__ >= '2.0.0':
    with open('./dataset/data/train.pkl', 'rb') as f:
        train_data = pickle.load(f)
    with open('./dataset/data/test.pkl', 'rb') as f:
        test_data = pickle.load(f)
else:
    with open('./dataset/data/train.pkl', 'rb') as f:
        train_data = NumpyCompatUnpickler(f).load()
    with open('./dataset/data/test.pkl', 'rb') as f:
        test_data = NumpyCompatUnpickler(f).load()


video_embeddings = train_data['video_embeddings']
audio_embeddings = train_data['audio_embeddings']


In [155]:
video_embeddings.shape

(29994, 1024)

In [156]:
audio_embeddings.shape

(29994, 768)

In [157]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AudioEmbed(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(768, 500),
            nn.LayerNorm(500),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(500, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, 256)
        )
    def forward(self, x):
        return F.normalize(self.model(x), dim=-1)

class VideoEmbed(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(1024, 768),
            nn.LayerNorm(768),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(768, 500),
            nn.LayerNorm(500),
            nn.GELU(),
            nn.Linear(500, 256)
        )
    def forward(self, x):
        return F.normalize(self.model(x), dim=-1)

In [158]:
device = "mps"

In [159]:
audio_embedder = AudioEmbed().to(device)
video_embedder = VideoEmbed().to(device)

batch_size = 200
opt_audio = torch.optim.Adam(audio_embedder.parameters())
opt_video = torch.optim.Adam(video_embedder.parameters())

loss_fn = nn.CosineSimilarity()

In [ ]:
import tqdm

def train_epoch(epoch):
    print("Epoch: ", epoch)

    loss_run = 0

    for i in tqdm.tqdm(range(0, len(audio_embeddings), batch_size)):
        Xaudio = torch.Tensor(audio_embeddings[i:i + batch_size]).to(device)
        Xvideo = torch.Tensor(video_embeddings[i:i + batch_size]).to(device)

        y_audio = audio_embedder(Xaudio)
        y_video = video_embedder(Xvideo)
        # print(loss_fn(y_audio, y_video))
        temperature = 0.05

        y_audio = F.normalize(y_audio, dim=-1)
        y_video = F.normalize(y_video, dim=-1)

        logits = y_audio @ y_video.T / temperature

        labels = torch.arange(len(y_audio)).to(device)

        loss_audio = F.cross_entropy(logits, labels)
        loss_video = F.cross_entropy(logits.T, labels)

        loss = (loss_audio + loss_video) / 2

        # loss = torch.ones(len(y_audio)) - loss_fn(y_audio, y_video)
        # loss = torch.mean(loss)

        loss_run += loss.item()
        
        opt_audio.zero_grad()
        opt_video.zero_grad()

        loss.backward()
        
        opt_audio.step()
        opt_video.step()
    print(loss_run)

In [163]:
for i in range(30): train_epoch(i)

Epoch:  0


100%|██████████| 150/150 [00:02<00:00, 74.03it/s]


1.9379843949282076
Epoch:  1


100%|██████████| 150/150 [00:01<00:00, 88.24it/s]


0.04562060083844699
Epoch:  2


...

0.0005338692135410383
Epoch:  28


100%|██████████| 150/150 [00:01<00:00, 87.30it/s]


0.0004979739485406753
Epoch:  29


100%|██████████| 150/150 [00:01<00:00, 89.36it/s]

0.0004570286027956172


In [164]:
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import cosine_distances
    
with open('./submission.csv', 'w') as f:
    f.write('subtaskID,sample_id,' + ','.join(f'{i}' for i in range(20)))
    for sample in test_data:
        audio_embeddings = sample['audio_embeddings']
        video_embeddings = sample['video_embeddings']
        sample_id = sample['sample_id']

        embed_audio = audio_embedder(torch.Tensor(audio_embeddings).to(device)).detach().cpu().numpy()
        embed_video = video_embedder(torch.Tensor(video_embeddings).to(device)).detach().cpu().numpy()

        idx, assig = linear_sum_assignment(cosine_distances(embed_audio, embed_video))

        predictions = assig
        ###

        f.write(f'\n1,{sample_id},' + ','.join(f'{predictions[i]}' for i in range(20)))

